# Table: Dataset Comparison

Generates the LaTeX table comparing EMERGE with other IE datasets.

**Two parts:**
1. **EMERGE stats** — computed from the complete dataset pkl (instances, relations, entities, triples)
2. **Other datasets** — computed by individual scripts in `other_datasets/` that download and analyze each dataset

The other dataset numbers are relatively stable (published datasets don't change),
so they can be hardcoded after running once. The EMERGE stats should be recomputed
if the dataset changes.

In [1]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.join(os.path.abspath('../../..'), 'src'))

## Compute EMERGE dataset stats

In [2]:
# --- CONFIGURE THIS ---

# Complete dataset (old pipeline pkl)
# TODO: replace with refactored pkl once full-dataset pipeline is ported
COMPLETE_DATASET_DIR = '/path/to/storage/emerge/output/experiments/dataset_stats_pkls/20260124_all_dataset_no_models_v13_all/'

In [3]:
df_gt = pd.read_pickle(os.path.join(COMPLETE_DATASET_DIR, 'df_wiki_predictions_and_gt_cie.pkl'))

n_instances = df_gt['hash_id'].nunique()
n_relations = df_gt['triple_relation'].nunique()
n_entities = pd.unique(df_gt[['triple_head', 'triple_tail']].to_numpy().ravel()).size
n_triples = df_gt[['hash_id', 'tkgu_type', 'triple_head', 'triple_relation', 'triple_tail']].drop_duplicates().shape[0]

print(f'EMERGE stats:')
print(f'  Instances: {n_instances:,} ({n_instances/1000:.2f}K)')
print(f'  Relations: {n_relations:,}')
print(f'  Entities:  {n_entities:,} ({n_entities/1000:.2f}K)')
print(f'  Triples:   {n_triples:,} ({n_triples/1000:.2f}K)')

EMERGE stats:
  Instances: 233,113 (233.11K)
  Relations: 833
  Entities:  174,610 (174.61K)
  Triples:   1,943,598 (1943.60K)


## Other dataset stats

These are from published datasets. Values computed by scripts in `other_datasets/`.
To recompute, run e.g.: `python other_datasets/webnlg_stats.py --lang en`

The numbers below are hardcoded from previous runs (datasets don't change).

In [4]:
# Hardcoded stats from other datasets (from running the scripts in other_datasets/)
# Format: (name, year, KG_evol, Text_evol, E, A, M+A, I, D, instances, relations, entities, triples)
other_datasets = [
    ('WebNLG',    '\\citeyear{gardent2017webnlg}',         False, False, True,  True,  False, False, False, 38870,   411,   3620,    115280),
    ('T-REX',     '\\citeyear{elsahar2019t}',              False, False, True,  True,  False, False, False, 6200000, 642,   4650000, 11000000),
    ('Wiki-NRE',  '\\citeyear{distiawan2019neural}',       False, False, True,  True,  False, False, False, 255650,  158,   279890,  330010),
    ('DocRED',    '\\citeyear{yao2019docred}',             False, False, True,  True,  False, False, False, 101870,  96,    782460,  1510000),
    ('BioRel',    '\\citeyear{xing2020biorel}',            False, False, True,  True,  False, False, False, 533560,  125,   17820,   578010),
    ('Wiki20',    '\\citeyear{han2020more}',               False, False, True,  True,  True,  False, False, 901310,  81,    392260,  901310),
    ('REBEL',     '\\citeyear{cabot2021rebel}',            False, False, True,  True,  False, False, False, 3060000, 1155,  5640000, 10310000),
    ('CDG',       '\\citeyear{zhang2022distant}',          False, False, True,  True,  False, False, False, 72330,   14,    15720,   175420),
    ('SynthIE-text', '\\citeyear{josifoski2023exploiting}', False, False, True,  True,  True,  False, False, 70270,   888,   121260,  241800),
    ('\\textsc{Text2KG}', '\\citeyear{mihindukulasooriya2023text2kgbench}', False, False, True, True, False, False, False, 18420, 291, 1910, 8620),
]

## Generate LaTeX table

In [5]:
def fmt_num(n):
    """Format number as X.XXK or X.XXM."""
    if n >= 1_000_000:
        return f'{n/1_000_000:.2f}M'
    elif n >= 1_000:
        return f'{n/1_000:.2f}K'
    else:
        return str(n)

def bool_to_latex(b):
    return '\\cmark' if b else '\\xmark'

# Build rows for other datasets
rows = []
for (name, year, kg, text, e, a, ma, i, d, inst, rels, ents, triples) in other_datasets:
    row = (
        f'{name} ({year}) & '
        f'{bool_to_latex(kg)} & {bool_to_latex(text)} & '
        f'{bool_to_latex(e)} & {bool_to_latex(a)} & {bool_to_latex(ma)} & '
        f'{bool_to_latex(i)} & {bool_to_latex(d)} & '
        f'{fmt_num(inst)} & {rels:,} & {fmt_num(ents)} & {fmt_num(triples)} \\\\'
    )
    rows.append(row)

# EMERGE row
emerge_row = (
    f'\\datasetname~(ours) & '
    f'\\cmark & \\cmark & '
    f'\\cmark & \\cmark & \\cmark & \\cmark & \\cmark & '
    f'{fmt_num(n_instances)} & {n_relations:,} & {fmt_num(n_entities)} & {fmt_num(n_triples)} \\\\'
)

body = '\n'.join(rows)

latex = f"""\\begin{{table*}}[t]
\\begin{{center}}
\\begin{{small}}
  \\begin{{sc}}
\\rowcolors{{6}}{{rowwhite}}{{rowgray}}
\\begin{{tabular}}{{l|cc|ccccc| cccc}}
\\toprule
& \\multicolumn{{2}}{{c|}}{{Evolution}}
& \\multicolumn{{5}}{{c|}}{{TKGU operations}}
& \\multicolumn{{4}}{{c}}{{Annotation statistics}} \\\\
\\cmidrule(lr){{2-3}}\\cmidrule(lr){{4-8}}\\cmidrule(lr){{9-12}}
Dataset & KG & Text & E & A & M+A & I & D & Inst. & Rels. & Ents. & Triples \\\\
\\midrule
{body}
\\midrule
{emerge_row}
\\bottomrule
\\end{{tabular}}
  \\end{{sc}}
\\end{{small}}
\\end{{center}}
\\vskip -0.1in
\\end{{table*}}"""

print(latex)

\begin{table*}[t]
\begin{center}
\begin{small}
  \begin{sc}
\rowcolors{6}{rowwhite}{rowgray}
\begin{tabular}{l|cc|ccccc| cccc}
\toprule
& \multicolumn{2}{c|}{Evolution}
& \multicolumn{5}{c|}{TKGU operations}
& \multicolumn{4}{c}{Annotation statistics} \\
\cmidrule(lr){2-3}\cmidrule(lr){4-8}\cmidrule(lr){9-12}
Dataset & KG & Text & E & A & M+A & I & D & Inst. & Rels. & Ents. & Triples \\
\midrule
WebNLG (\citeyear{gardent2017webnlg}) & \xmark & \xmark & \cmark & \cmark & \xmark & \xmark & \xmark & 38.87K & 411 & 3.62K & 115.28K \\
T-REX (\citeyear{elsahar2019t}) & \xmark & \xmark & \cmark & \cmark & \xmark & \xmark & \xmark & 6.20M & 642 & 4.65M & 11.00M \\
Wiki-NRE (\citeyear{distiawan2019neural}) & \xmark & \xmark & \cmark & \cmark & \xmark & \xmark & \xmark & 255.65K & 158 & 279.89K & 330.01K \\
DocRED (\citeyear{yao2019docred}) & \xmark & \xmark & \cmark & \cmark & \xmark & \xmark & \xmark & 101.87K & 96 & 782.46K & 1.51M \\
BioRel (\citeyear{xing2020biorel}) & \xmark & \xmark & \cm